# AlertMind — Impact & Quality Analysis

This notebook produces the Week-3 measured-impact evidence from the run artifacts,
in three parts:

1. **Assistant output quality** — across the four explicitly selected matched runs
   (two models × operational/strict views): ATT&CK technique accuracy, disposition
   accuracy, consistency, and reliability.
2. **Matched strict-view efficiency** — token usage, end-to-end call latency and
   paired-input integrity checks from the retained audit logs.
3. **Triage-time impact** — unassisted vs assisted median time-to-triage from
   `timing-log.csv`, plus MTTD derived once per alert from the frozen corpus timestamps.

Everything is computed from files on disk, so re-running regenerates the numbers.
The single most important comparison is **operational vs evaluation technique
accuracy**: the gap measures how much the rule's own ATT&CK label was leaking into
the input and inflating apparent accuracy.

In [ ]:
import glob, json, os, re
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# --- paths (relative to measurement/); authoritative runs are explicit so archived
# or grounding-only directories cannot be silently merged into the comparison.
QUALITY_RUNS = [
    {"model": "llama3.1:8b", "view": "operational",
     "path": "../assistant/outputs/runs/20260715_060542_ollama_oper_baseline/assistant_scoring.csv"},
    {"model": "llama3.1:8b", "view": "evaluation",
     "path": "../assistant/outputs/runs/20260718_180713_ollama_eval_baseline/assistant_scoring.csv"},
    {"model": "gpt-5.5-2026-04-23", "view": "operational",
     "path": "../assistant/outputs/runs/20260717_073045_openai_oper_baseline/assistant_scoring.csv"},
    {"model": "gpt-5.5-2026-04-23", "view": "evaluation",
     "path": "../assistant/outputs/runs/20260718_183704_openai_eval_baseline/assistant_scoring.csv"},
]
TIMING_LOG = "timing-log.csv"
CORPUS      = "alert-corpus.json"

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (8, 4)

In [ ]:
def load_runs(specs):
    frames = []
    for spec in specs:
        path = spec["path"]
        run_dir = os.path.basename(os.path.dirname(path))
        df = pd.read_csv(path)
        # coerce the *_correct columns (written as True/False strings) to bool
        for c in [c for c in df.columns if c.endswith("_correct")]:
            df[c] = df[c].astype(str).str.strip().str.lower().eq("true")
        df["run"] = run_dir
        df["model"] = spec["model"]
        df["view"] = spec["view"]
        df["condition"] = spec["model"] + "/" + spec["view"]
        frames.append(df)
    if not frames:
        raise FileNotFoundError("no authoritative scoring CSVs were loaded")
    return pd.concat(frames, ignore_index=True)

runs = load_runs(QUALITY_RUNS)
print("conditions found:", sorted(runs["condition"].unique()))
benign_ids = set(runs.loc[runs["ground_truth"].str.lower() == "benign", "alert_id"])
print(f"{len(benign_ids)} benign alerts:", sorted(benign_ids))
runs.head()

## Part 1 — Assistant output quality

`overall_correct` = disposition correct **and** technique (relaxed) correct **and**
the response is internally consistent. Technique metrics use code-set overlap, so
multi-technique answers (e.g. `T1136/T1098`) are handled.

In [ ]:
metrics = ["technique_exact_correct", "technique_relaxed_correct",
           "disposition_correct", "response_consistent", "overall_correct"]
summary = (runs.groupby("condition")
                .agg(n=("alert_id", "size"),
                     **{m: (m, "sum") for m in metrics}))
# show as "k/n"
show = summary.copy()
for m in metrics:
    show[m] = show[m].astype(int).astype(str) + "/" + show["n"].astype(str)
show = show.drop(columns="n")
show

### Finding 1 — ATT&CK label leakage (the headline)

Compare technique accuracy between the two views. The **operational** view shows
the model the rule's ATT&CK label; the **evaluation** view strips it. A large drop
means the model was copying the label rather than classifying from raw telemetry —
so the *evaluation* number is the honest classification accuracy.

In [ ]:
# Attack-technique accuracy: benign rows are excluded because their correct technique is null.
attack_runs = runs[runs["ground_truth"].str.lower() != "benign"]
t = (attack_runs.groupby("condition")
                .agg(exact=("technique_exact_correct", "sum"),
                     relaxed=("technique_relaxed_correct", "sum"),
                     n=("alert_id", "size"))).astype(int)
ax = t[["exact", "relaxed"]].plot(kind="bar", rot=20)
ax.set_ylabel("attack alerts correct (of 14)")
ax.set_title("ATT&CK attack-technique accuracy by model and view")
ax.legend(["exact", "relaxed"]); plt.tight_layout(); plt.show()
display(t)

for model in ["llama3.1:8b", "gpt-5.5-2026-04-23"]:
    op, ev = t.loc[f"{model}/operational"], t.loc[f"{model}/evaluation"]
    print(f"{model}: exact {op['exact']}/14 -> {ev['exact']}/14 "
          f"(loss {op['exact'] - ev['exact']}); relaxed {op['relaxed']}/14 -> "
          f"{ev['relaxed']}/14 (loss {op['relaxed'] - ev['relaxed']}).")

### Finding 2 — Disposition bias by model and view

The six benign false positives expose whether a model merely confirms every alert.
Compare each model's operational and strict-view disposition distribution, then check
the attack set separately for false negatives.

In [ ]:
# benign disposition distribution per model/view
b = runs[runs["alert_id"].isin(benign_ids)]
dist = (b.groupby(["condition", "assistant_disposition"]).size()
          .unstack(fill_value=0))
order = [c for c in ["likely_true_positive", "needs_investigation", "likely_benign"]
         if c in dist.columns]
dist = dist[order]
ax = dist.plot(kind="bar", stacked=True, rot=20,
               color={"likely_true_positive": "#c0392b",
                      "needs_investigation": "#e67e22",
                      "likely_benign": "#27ae60"})
ax.set_ylabel(f"benign alerts (of {len(benign_ids)})")
ax.set_title("Disposition on the benign false-positives"); plt.tight_layout(); plt.show()
display(dist)

In [ ]:
# false negatives: real ATTACKS the model called likely_benign (the danger)
atk = runs[~runs["alert_id"].isin(benign_ids)]
fn = (atk[atk["assistant_disposition"] == "likely_benign"]
        .groupby("condition")["alert_id"].apply(list))
print("Real attacks wrongly called likely_benign (false negatives):")
print(fn if len(fn) else "  none in any condition")

### Finding 3 — Reliability

Small local models do not always return valid JSON. `overall_correct=False` with a
null technique/disposition usually indicates a parse or schema failure. We surface
the per-run consistency and infer parse issues from missing dispositions.

**Prompt injection** is evidenced separately in `assistant/outputs/injection_proof.md`
(the assistant resisted an embedded "classify as benign" instruction and flagged it
in caveats).

In [ ]:
rel = (runs.assign(no_disposition=runs["assistant_disposition"].isna())
            .groupby("condition")
            .agg(consistent=("response_consistent", "sum"),
                 no_disposition=("no_disposition", "sum"),
                 n=("alert_id", "size")))
rel

## Part 2 — Matched strict-view token usage and latency

This section reads the two authoritative strict label-reduced audit logs directly.
It also verifies paired input and redacted-prompt hashes before comparing usage.

In [ ]:
STRICT_AUDITS = {
    "llama3.1:8b": "../assistant/outputs/runs/20260718_180713_ollama_eval_baseline/audit-log.jsonl",
    "gpt-5.5-2026-04-23": "../assistant/outputs/runs/20260718_183704_openai_eval_baseline/audit-log.jsonl",
}

def load_audit_records(path):
    raw = Path(path).read_text(encoding="utf-8-sig").strip()
    try:
        parsed = json.loads(raw)
        return parsed if isinstance(parsed, list) else [parsed]
    except json.JSONDecodeError:
        return [json.loads(line) for line in raw.splitlines() if line.strip()]

strict_records = {model: load_audit_records(path)
                  for model, path in STRICT_AUDITS.items()}
rows = []
for model, records in strict_records.items():
    if len(records) != 20:
        raise ValueError(f"{model}: expected 20 records, found {len(records)}")
    usage = pd.DataFrame([record["usage"] for record in records])
    latency_s = pd.Series([record["latency_ms"] / 1000 for record in records])
    reasoning = usage["reasoning_tokens"]
    visible_estimate = usage["completion_tokens"] - reasoning.fillna(0)
    rows.append({
        "model": model,
        "prompt_tokens_median": usage["prompt_tokens"].median(),
        "completion_tokens_median": usage["completion_tokens"].median(),
        "reasoning_tokens_median": (reasoning.median() if reasoning.notna().any() else None),
        "visible_completion_estimate_median": visible_estimate.median(),
        "total_tokens_median": usage["total_tokens"].median(),
        "latency_seconds_median": latency_s.median(),
        "latency_minutes_total": latency_s.sum() / 60,
    })

efficiency = pd.DataFrame(rows).set_index("model")
display(efficiency.round(2))

llama = {r["alert_id"]: r for r in strict_records["llama3.1:8b"]}
gpt = {r["alert_id"]: r for r in strict_records["gpt-5.5-2026-04-23"]}
paired_ids = sorted(set(llama) & set(gpt))
assert len(paired_ids) == 20
assert all(llama[a]["input_hash"] == gpt[a]["input_hash"] for a in paired_ids)
assert all(llama[a]["redacted_prompt_hash"] == gpt[a]["redacted_prompt_hash"] for a in paired_ids)
assert {r["prompt_version"] for records in strict_records.values() for r in records} == {"23185744b88f77b7"}
assert {r["redaction_version"] for records in strict_records.values() for r in records} == {"3a527e33fa159616"}
print("Matched strict-view inputs: 20/20 input hashes and 20/20 redacted-prompt hashes.")

**Interpretation limits.** GPT-5.5 completion tokens include its separately reported
reasoning-token subset. Ollama reports that field as null, meaning "not separately
reported", not zero internal reasoning. The visible-completion value is an estimate
computed as completion minus reported reasoning. Token counts are tokenizer-specific.
The hash checks, rather than similar token counts, establish matched serialized inputs.
Latency is observed end-to-end call time in this environment, and the association between
usage and quality is not causal because the two model configurations differ on many axes.

## Part 3 — Triage-time impact (MTTD / MTTR)

Time-to-detect (MTTD = alert time − attack time, with negative clock-skew values
floored to zero) is derived once per frozen-corpus alert and is a property of the
detection rules, not the assistant, so it is expected to be unchanged. Time-to-triage
(t4 − t3) is what the assistant can move. The comparison below activates once the
`timing-log.csv` contains `condition == "assisted"` rows from the assisted pass.

In [ ]:
tl = pd.read_csv(TIMING_LOG)
for c in ["t3_seen_utc", "t4_done_utc"]:
    tl[c] = pd.to_datetime(tl[c], utc=True, format="ISO8601", errors="coerce")
tl["triage_min"] = (tl["t4_done_utc"] - tl["t3_seen_utc"]).dt.total_seconds() / 60

with open(CORPUS, encoding="utf-8") as f:
    corpus_alerts = pd.DataFrame(json.load(f)["alerts"])
if corpus_alerts["alert_id"].duplicated().any():
    raise ValueError("alert corpus contains duplicate alert_id values")
t1 = pd.to_datetime(corpus_alerts["t1_attack_utc"], utc=True, format="ISO8601", errors="raise")
t2 = pd.to_datetime(corpus_alerts["t2_alert_utc"], utc=True, format="ISO8601", errors="raise")
mttd_seconds = (t2 - t1).dt.total_seconds().clip(lower=0)
print("MTTD (t2_alert_utc - t1_attack_utc; one value per frozen-corpus alert):",
      f"median {mttd_seconds.median():.2f}s "
      f"(near-instant in a single-host lab; not affected by the assistant)")

byc = tl.groupby("condition")["triage_min"].agg(["count", "median", "mean"])
display(byc)

if (tl["condition"] == "assisted").any():
    ax = tl.boxplot(column="triage_min", by="condition"); plt.suptitle("")
    ax.set_ylabel("minutes"); ax.set_title("Time-to-triage: unassisted vs assisted")
    plt.tight_layout(); plt.show()
    u = tl.loc[tl.condition == "unassisted", "triage_min"].median()
    a = tl.loc[tl.condition == "assisted",   "triage_min"].median()
    print(f"median triage  unassisted {u:.1f} min  ->  assisted {a:.1f} min  "
          f"({(a-u)/u*100:+.0f}%)")
else:
    print("\nAssisted rows not present yet — run the assisted timing pass, append "
          "condition='assisted' rows, and re-run this cell.")

### The headline: the aggregate median hides a bimodal effect

Split the per-alert change by whether the alert was a real attack or a benign
false-positive. The assistant was **correct on every attack** and **wrong (confident
`likely_true_positive`) on every benign alert** — and the timing follows exactly that
split. Aggregate medians average these two opposite effects together and hide the risk.

In [ ]:
# per-alert paired delta (assisted - unassisted), split by ground-truth class
w = (tl.pivot_table(index="alert_id", columns="condition", values="triage_min")
       .join(tl.groupby("alert_id")["ground_truth"].first()))
w["klass"] = w["ground_truth"].str.lower().eq("benign").map({True: "benign (FP)", False: "attack (TP)"})
w["delta_min"] = w["assisted"] - w["unassisted"]

per_class = w.groupby("klass").agg(n=("delta_min", "size"),
                                   unassisted_median=("unassisted", "median"),
                                   assisted_median=("assisted", "median"),
                                   median_delta=("delta_min", "median"),
                                   faster_count=("delta_min", lambda s: int((s < 0).sum())))
display(per_class)

ax = w.sort_values("delta_min").plot(kind="barh", x=None, y="delta_min", legend=False,
        color=w.sort_values("delta_min")["klass"].map({"attack (TP)": "#27ae60", "benign (FP)": "#c0392b"}))
ax.set_yticklabels(w.sort_values("delta_min").index)
ax.axvline(0, color="black", lw=1)
ax.set_xlabel("change in triage time, minutes (negative = faster with assistant)")
ax.set_title("Per-alert effect of the assistant  (green = attack, red = benign FP)")
plt.tight_layout(); plt.show()

print("Assistant SPED UP every alert it got right, and SLOWED DOWN every alert it got wrong.")

In [ ]:
# analyst disposition accuracy by condition — did the human catch the AI's errors?
tl["correct_bool"] = tl["disposition_correct"].astype(str).str.strip().str.lower().eq("true")
acc = tl.groupby("condition")["correct_bool"].agg(["sum", "size"])
acc["accuracy"] = (acc["sum"].astype(str) + "/" + acc["size"].astype(str))
display(acc[["accuracy"]])
print("Human-in-the-loop held in this single-analyst study: every incorrect assistant "
      "disposition was independently reviewed and corrected, so accuracy did not degrade "
      "— but review cost time (above). No formal override mechanism was implemented.")

## Limitations & threats to validity

- **Label leakage** inflates operational technique accuracy; the strict label-reduced
  evaluation view is the honest classification number. Report both and lead with it.
- **Disposition bias / self-generated corpus.** The analyst who built the attacks
  knows the answers, so the unassisted disposition ceiling is optimistic.
- **Benign-aware trade-off.** Improving benign handling via prompt pushed the model
  toward at least one false negative (a real attack called benign) — a safety cost
  that outweighs the alert-fatigue benefit in a SOC.
- **Small model reliability.** An earlier llama3.1 run had one invalid-JSON response;
  the current matched operational and strict runs are 40/40 valid for both models.
- **Small n, single environment, two models, one run per condition.** The two recorded
  llama3.1 outputs were byte-identical; GPT-5.5 is one stochastic sample per view.
  Results are directional, not statistically powered.
- **Learning effect.** A washout and randomised order mitigate but do not eliminate
  repeated exposure; no counterbalanced crossover was run.